# Cardiothoracic Abnormality Assessment with KaleCardiac

Classify **Healthy** versus **Cardiothoracic Abnormality** from a paired chest X-ray (CXR)
and twelve-lead ECG with **CardioVAE**: a multimodal variational autoencoder pretrained on
unlabelled pairs, then fine-tuned on a small labelled set.

This notebook follows PyKale's
[cardiothoracic abnormality assessment tutorial](https://pykale.github.io/mmai-tutorials/tutorials/cardiac-abnormality-assessment/tutorial-heart.html)
step for step, on the same data, with the CardioVAE pipeline that has moved from PyKale
into **KaleCardiac**. Each step is a few lines of the KaleCardiac API:

| Step | KaleCardiac API |
| --- | --- |
| 1. Load data | `loaddata.MultimodalDataset.from_arrays`, `loaddata.HoldOutSplit` |
| 2. Define the model | `model.embed.CardioVAE` |
| 3. Pretrain | `pipeline.ReconstructionTask`, `pipeline.CardiacTrainer` |
| 3. Fine-tune | `CardioVAE.load_pykale_state_dict`, `model.embed.MultimodalPredictor`, `pipeline.ClassificationTask` |
| 4. Evaluate | `evaluate.predict_split`, `evaluate.binary_metrics` |
| 5. Interpret | `interpret.modality_attributions`, `interpret.normalize_attribution` |

**Runtime:** about 10 minutes on a GPU (a Colab T4), as for the PyKale tutorial.

**KaleCardiac ships no data.** The tutorial data is a preprocessed subset of MIMIC-CXR and
MIMIC-IV-ECG, shared on Google Drive for the workshop (see Step 1). Without it, the
notebook generates small synthetic stand-ins of the same layout so that every cell runs --
a check that the workflow executes, **not a result**.

## Step 0: Environment Preparation

On **Google Colab**, set the runtime to a GPU (*Runtime → Change runtime type → T4 GPU*),
then run the two cells below: one installs KaleCardiac, the other connects your Google
Drive, where the data and the pretrained checkpoint live. Running locally, skip both and
install KaleCardiac with `pip install "kalecardiac[interpret]"`.

In [ ]:
%%capture
!pip install --quiet "kalecardiac[interpret] @ git+https://github.com/pykale/cardiac@main"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Warnings and PyTorch Lightning's progress messages are silenced, so the output shows what
matters.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore")
for name in ("pytorch_lightning", "lightning.pytorch"):
    logging.getLogger(name).setLevel(logging.ERROR)

## Configuration

KaleCardiac's configuration schema (`kalecardiac.config`) holds every setting the PyKale
tutorial tunes, so the pretraining and fine-tuning configurations are two copies of it.
`DATASET.ROOT` points at the tutorial data on your Drive.

| PyKale tutorial | KaleCardiac |
| --- | --- |
| `TRAIN.LAMBDA_IMAGE`, `TRAIN.LAMBDA_SIGNAL` | `OBJECTIVE.RECONSTRUCTION_WEIGHTS` |
| `TRAIN.SCALE_FACTOR` | `OBJECTIVE.SCALE_FACTOR` |
| `TRAIN.EPOCHS`, `TRAIN.LR`, `DATA.BATCH_SIZE` | `SOLVER.MAX_EPOCHS`, `SOLVER.BASE_LR`, `SOLVER.BATCH_SIZE` |
| `FT.HIDDEN_DIM` | `MODEL.HEAD_HIDDEN` |

In [ ]:
from kalecardiac.config import get_cfg_defaults

# Pre-training configuration.
cfg_pt = get_cfg_defaults()
cfg_pt.DATASET.ROOT = "/content/drive/MyDrive/EMBC_workshop_data"
cfg_pt.SOLVER.SEED = 123
cfg_pt.OBJECTIVE.SCALE_FACTOR = 1e-4
cfg_pt.OBJECTIVE.UNIMODAL_STREAMS = True  # the tri-stream objective
# ------ Hyperparameters to play with -----
cfg_pt.MODEL.LATENT_DIM = 128
cfg_pt.SOLVER.MAX_EPOCHS = 1
cfg_pt.SOLVER.BASE_LR = 1e-3
cfg_pt.SOLVER.BATCH_SIZE = 128
cfg_pt.OBJECTIVE.RECONSTRUCTION_WEIGHTS = ["image:1", "signal:10"]
cfg_pt.OBJECTIVE.ANNEALING_EPOCHS = cfg_pt.SOLVER.MAX_EPOCHS  # the KL weight rises over training

# Fine-tuning configuration.
cfg_ft = get_cfg_defaults()
cfg_ft.DATASET.ROOT = cfg_pt.DATASET.ROOT
cfg_ft.DATASET.VAL_RATIO = 0.2
cfg_ft.SOLVER.SEED = 42
cfg_ft.MODEL.LATENT_DIM = 256  # the width of the published checkpoint
# ------ Hyperparameters to play with -----
cfg_ft.SOLVER.MAX_EPOCHS = 10
cfg_ft.SOLVER.BASE_LR = 1e-3
cfg_ft.SOLVER.BATCH_SIZE = 32
cfg_ft.MODEL.HEAD_HIDDEN = [128]

print(cfg_pt.OBJECTIVE, cfg_ft.SOLVER, sep="\n\n")

## Step 1: Data Loading and Preparation

Each subject is a pair: a twelve-lead ECG and a chest X-ray, preprocessed as the PyKale
tutorial did -- each lead normalized, the twelve leads interleaved sample by sample into
one 60,000-sample strip (ten seconds at 500 Hz), and each radiograph resized to 224×224.

**The data.** Add the shared
[`EMBC_workshop_data`](https://drive.google.com/drive/folders/1N7-fMWsdK-tuB76SdC-GF1njYYGx0Z-i?usp=sharing)
folder to *My Drive* as a shortcut (*Organize → Add shortcut → My Drive*). It holds the
first 1,000 pairs for pretraining, the last 1,000 with labels for fine-tuning, and the
published CardioVAE checkpoint. The two subsets do not overlap.

**From raw files instead.** With PhysioNet access to the full MIMIC data, build the
sources from one file per subject -- `ECGFileSource` with `get_ecg_transform`, and
`ImageFileSource` with `get_image_transform`, both in `kalecardiac.loaddata` and
`kalecardiac.prepdata` -- and pass them to `MultimodalDataset`. Everything after this
step is unchanged.

In [ ]:
from pathlib import Path

import numpy as np
import torch

DATA = Path(cfg_pt.DATASET.ROOT)
SYNTHETIC = not (DATA / "ecg_features_tensor_1000.pt").exists()
NUM_LEADS, SAMPLING_RATE = 12, 500  # the tutorial data: ten seconds at 500 Hz, interleaved


def stand_ins(num_subjects: int, seed: int):
    """ECGs, radiographs and labels of the tutorial's layout, smaller, with a label to learn."""
    generator = torch.Generator().manual_seed(seed)
    severity = torch.rand(num_subjects, generator=generator)
    time = torch.arange(10 * SAMPLING_RATE) / SAMPLING_RATE
    beat = torch.sin(2 * torch.pi * (1.0 + severity[:, None]) * time)  # a heart rate that rises with severity
    noise = torch.randn(num_subjects, time.numel(), NUM_LEADS, generator=generator)
    ecg = (beat[:, :, None] * torch.linspace(0.5, 1.5, NUM_LEADS) + noise).reshape(num_subjects, 1, -1)  # interleaved
    cxr = torch.rand(num_subjects, 1, 64, 64, generator=generator) + 0.15 * severity[:, None, None, None]
    return ecg, cxr, (severity > 0.5).long().numpy()


if SYNTHETIC:
    SAMPLING_RATE = 50  # ten seconds at 50 Hz keeps the stand-in model small enough for a CPU
    print(
        "The tutorial data was not found, so synthetic stand-ins are used: this checks the workflow, it is not a result."
    )

### Pre-training data

The pretraining pairs carry no label. `MultimodalDataset.from_arrays` turns the two
tensors into a dataset whose modalities are named `signal` and `image` -- the names
`CardioVAE` uses -- and 80% train the model while 20% monitor it.

In [ ]:
from functools import partial

from torch.utils.data import DataLoader, Subset, random_split

from kalecardiac.loaddata import MultimodalDataset, collate_subjects

if SYNTHETIC:
    ecg_pt, cxr_pt, _ = stand_ins(300, seed=cfg_pt.SOLVER.SEED)
else:
    ecg_pt = torch.load(DATA / "ecg_features_tensor_1000.pt", map_location="cpu")  # (1000, 1, 60000)
    cxr_pt = torch.load(DATA / "cxr_features_tensor_1000.pt", map_location="cpu")  # (1000, 1, 224, 224)

pretrain_set = MultimodalDataset.from_arrays({"signal": ecg_pt, "image": cxr_pt})
train_set_pt, val_set_pt = random_split(
    pretrain_set, [0.8, 0.2], generator=torch.Generator().manual_seed(cfg_pt.SOLVER.SEED)
)

loader = partial(DataLoader, collate_fn=collate_subjects, num_workers=cfg_pt.DATASET.NUM_WORKERS)
train_loader_pt = loader(train_set_pt, batch_size=cfg_pt.SOLVER.BATCH_SIZE, shuffle=True)
val_loader_pt = loader(val_set_pt, batch_size=cfg_pt.SOLVER.BATCH_SIZE)
print(f"pretraining: {len(train_set_pt)} pairs to train on, {len(val_set_pt)} to validate")

### Fine-tuning data

The fine-tuning pairs carry a label: **0** Healthy, **1** Cardiothoracic Abnormality.
`HoldOutSplit` holds 20% out for validation, stratified so both halves keep the
cohort's class balance.

In [ ]:
import pandas as pd

from kalecardiac.loaddata import HoldOutSplit

if SYNTHETIC:
    ecg_ft, cxr_ft, labels = stand_ins(300, seed=cfg_ft.SOLVER.SEED)
else:
    ecg_ft = torch.load(DATA / "ecg_features_tensor_last_1000.pt", map_location="cpu")
    cxr_ft = torch.load(DATA / "cxr_features_tensor_last_1000.pt", map_location="cpu")
    labels = pd.read_csv(DATA / "chexpert_healthy_abnormality_subset.csv")["label"].to_numpy()

finetune_set = MultimodalDataset.from_arrays({"signal": ecg_ft, "image": cxr_ft}, targets={"label": labels})
splitter = HoldOutSplit(
    test_size=cfg_ft.DATASET.VAL_RATIO, val_size=0.0, stratify_by=["label"], random_state=cfg_ft.SOLVER.SEED
)
split = next(splitter.split(pd.DataFrame({"label": labels})))

train_loader_ft = loader(Subset(finetune_set, split["train"]), batch_size=cfg_ft.SOLVER.BATCH_SIZE, shuffle=True)
val_loader_ft = loader(Subset(finetune_set, split["test"]), batch_size=cfg_ft.SOLVER.BATCH_SIZE)
print(f"fine-tuning: {len(split['train'])} labelled pairs to train on, {len(split['test'])} to validate")

## Step 2: Model Definition

CardioVAE is one KaleCardiac model, `CardioVAE`, assembled from the stages of the API:

**Embed.** One variational encoder per modality maps its input to a Gaussian over a shared
latent space -- a 1-D convolutional encoder for the ECG and a 2-D one for the CXR -- and a
**product of experts** fuses them, together with a standard-normal prior expert, into one
joint posterior. A subject missing a modality simply contributes no expert.

**Predict.** For pretraining, a decoder per modality reconstructs both inputs from a sample
of the joint posterior. For fine-tuning, the frozen encoders' posterior means are
concatenated and a small classification head reads them.

| In the PyKale tutorial | In KaleCardiac |
| --- | --- |
| `kale.embed.signal_cnn.SignalVAEEncoder` | `model.layers.Conv1dEncoder` in a `model.embed.VariationalEncoder` |
| `kale.embed.image_cnn.ImageVAEEncoder` | `model.layers.Conv2dEncoder` in a `model.embed.VariationalEncoder` |
| `kale.embed.multimodal_fusion.ProductOfExperts` | `model.layers.product_of_experts`, which calls it |
| `kale.predict.decode.SignalVAEDecoder`, `ImageVAEDecoder` | `model.layers.Conv1dDecoder`, `Conv2dDecoder` |
| `kale.embed.multimodal_encoder.SignalImageVAE` | `model.embed.CardioVAE` |
| `kale.predict.decode.SignalImageFineTuningClassifier` | `model.embed.MultimodalPredictor` over `CardioVAE.latent_embedders()` |
| `kale.pipeline.multimodal_trainer.SignalImageTriStreamVAETrainer` | `pipeline.CardiacTrainer` with `pipeline.ReconstructionTask` |
| `kale.pipeline.multimodal_trainer.SignalImageFineTuningTrainer` | `pipeline.CardiacTrainer` with `pipeline.ClassificationTask` |

With its default sizes `CardioVAE` is PyKale's `SignalImageVAE` parameter for parameter;
here the sizes come from the data.

In [ ]:
from kalecardiac.model.embed import CardioVAE

model = CardioVAE(
    signal_length=ecg_pt.shape[-1], image_size=tuple(cxr_pt.shape[-2:]), latent_dim=cfg_pt.MODEL.LATENT_DIM
)
print(f"CardioVAE with {sum(p.numel() for p in model.parameters()):,} parameters")

## Step 3: Model Training

### Multimodal pretraining

The **tri-stream** objective is three ELBOs: one through the joint posterior of both
modalities, and one through each modality alone, so that each encoder stays usable when the
other modality is absent. The ECG reconstruction is weighted ten times the radiograph, and
the KL term is annealed in. One trainer, `CardiacTrainer`, trains any model for any task;
pretraining is a `ReconstructionTask`, which needs no labels. About two minutes per epoch on
the tutorial data.

In [ ]:
import pytorch_lightning as pl

from kalecardiac.config import parse_weights
from kalecardiac.pipeline import CardiacTrainer, ReconstructionTask
from kalecardiac.utils import set_seed

set_seed(cfg_pt.SOLVER.SEED)
objective = ReconstructionTask(
    weights=parse_weights(cfg_pt.OBJECTIVE.RECONSTRUCTION_WEIGHTS),
    scale_factor=cfg_pt.OBJECTIVE.SCALE_FACTOR,
    annealing_epochs=cfg_pt.OBJECTIVE.ANNEALING_EPOCHS,
    unimodal_streams=cfg_pt.OBJECTIVE.UNIMODAL_STREAMS,
)
pretrainer = CardiacTrainer(model, objective, max_epochs=cfg_pt.SOLVER.MAX_EPOCHS, init_lr=cfg_pt.SOLVER.BASE_LR)

trainer = pl.Trainer(
    max_epochs=cfg_pt.SOLVER.MAX_EPOCHS,
    accelerator=cfg_pt.SOLVER.ACCELERATOR,
    devices=cfg_pt.SOLVER.DEVICES,
    log_every_n_steps=10,
    enable_checkpointing=False,
    logger=False,
)
trainer.fit(pretrainer, train_loader_pt, val_loader_pt)

torch.save(model.state_dict(), "cardiovae_pretrained.pt")
print("validation ELBO:", float(trainer.callback_metrics["valid_loss"]))

### Multimodal fine-tuning

Fine-tuning starts from the **published CardioVAE**, pretrained on about 50,000 pairs.
`load_pykale_state_dict` reads the checkpoint PyKale's tutorial loads, translating its
parameter names. (Without the checkpoint, the representation pretrained above is used.)
The encoders are frozen, and a head with one hidden layer of 128 units learns the endpoint.
Any other endpoint is the same few lines with another task -- a `RegressionTask` predicts a
measurement instead. About two minutes for ten epochs.

In [ ]:
from kalecardiac.model.embed import MultimodalPredictor
from kalecardiac.pipeline import ClassificationTask

checkpoint = DATA / "CardioVAE.pth"
if checkpoint.exists():
    cardiovae = CardioVAE(latent_dim=cfg_ft.MODEL.LATENT_DIM)  # 60,000-sample ECG, 224x224 CXR: the published sizes
    cardiovae.load_pykale_state_dict(torch.load(checkpoint, map_location="cpu"))
else:
    cardiovae = model

set_seed(cfg_ft.SOLVER.SEED)
task = ClassificationTask(hidden_dims=cfg_ft.MODEL.HEAD_HIDDEN, dropout=cfg_ft.MODEL.DROPOUT)
predictor = MultimodalPredictor(cardiovae.latent_embedders(frozen=cfg_ft.MODEL.FREEZE_ENCODERS), task.build_head)
classifier = CardiacTrainer(predictor, task, max_epochs=cfg_ft.SOLVER.MAX_EPOCHS, init_lr=cfg_ft.SOLVER.BASE_LR)

trainer = pl.Trainer(
    max_epochs=cfg_ft.SOLVER.MAX_EPOCHS,
    accelerator=cfg_ft.SOLVER.ACCELERATOR,
    devices=cfg_ft.SOLVER.DEVICES,
    log_every_n_steps=10,
    enable_checkpointing=False,
    logger=False,
)
trainer.fit(classifier, train_loader_ft, val_loader_ft)

## Step 4: Evaluation

The fine-tuned model scores every validation subject, and `binary_metrics` summarizes the
scores: accuracy, the area under the ROC curve (AUROC) and the Matthews correlation
coefficient (MCC), as the PyKale tutorial reports, with the sensitivity and specificity a
clinical reader asks for. Predictions stay per subject, so they can be re-scored at another
threshold.

In [ ]:
from kalecardiac.evaluate import binary_metrics, predict_split

predictions = predict_split(classifier, val_loader_ft, split="val")
metrics = binary_metrics(predictions.targets["label"], predictions.scores)

summary = pd.Series({name: metrics[name] for name in ("accuracy", "roc_auc", "mcc", "sensitivity", "specificity")})
summary.rename({"roc_auc": "AUROC", "mcc": "MCC"}).round(3).to_frame("validation")

## Step 5: Interpretation

Integrated gradients attribute the prediction for one subject to every sample of its ECG
and every pixel of its CXR. `normalize_attribution` rescales each map to [0, 1], and the
regions above a threshold are drawn. The strip interleaves the twelve leads, so one lead is
read out of it by name, in the order MIMIC-IV-ECG stores them. About ten seconds.

In [ ]:
from scipy.ndimage import binary_dilation

from kalecardiac.interpret import modality_attributions, normalize_attribution
from kalecardiac.loaddata import get_lead_indices

# The order MIMIC-IV-ECG stores its leads in, read from its WFDB headers: aVF comes
# before aVL, unlike the clinical order. The tutorial data keeps it.
MIMIC_LEADS = ["I", "II", "III", "aVR", "aVF", "aVL", "V1", "V2", "V3", "V4", "V5", "V6"]

# User can change these to interpret another subject, lead or threshold.
SAMPLE_IDX, LEAD = 3, "II"
ECG_THRESHOLD, CXR_THRESHOLD, ZOOM_RANGE = 0.75, 0.75, (3.5, 4.0)

sample = collate_subjects([val_loader_ft.dataset[SAMPLE_IDX]])
attributions = modality_attributions(classifier.model, sample.modalities, present=sample.present)

# The strip interleaves the twelve leads sample by sample; one column per lead.
lead = get_lead_indices(MIMIC_LEADS, [LEAD])[0]
ecg = sample.modalities["signal"][0, 0].numpy().reshape(-1, NUM_LEADS)[:, lead]
ecg_importance = normalize_attribution(attributions["signal"][0, 0]).reshape(-1, NUM_LEADS)[:, lead]
time = np.arange(ecg.size) / SAMPLING_RATE
cxr = sample.modalities["image"][0, 0].numpy()
cxr_importance = normalize_attribution(attributions["image"][0, 0])

probability = torch.sigmoid(classifier.predict(sample).prediction).item()
print(
    f"subject {SAMPLE_IDX}: label {int(sample.target['label'])}, predicted probability of abnormality {probability:.2f}"
)

**Full ECG view.** The whole ten seconds of one lead, with the most important regions in
red.

In [ ]:
import matplotlib.pyplot as plt


def plot_ecg(window: slice, title: str, width: float = 1):
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(time[window], ecg[window], color="black", linewidth=width, label="ECG waveform")
    important = np.where(ecg_importance[window] >= ECG_THRESHOLD)[0]
    for index in important:
        stretch = slice(max(index - 2, 0), index + 3)
        ax.plot(time[window][stretch], ecg[window][stretch], color="red", linewidth=2 * width)
    ax.set_xlabel("Time (seconds)", fontsize="x-large")
    ax.set_title(title, fontsize="x-large")
    ax.legend(["ECG waveform", "Important regions"], fontsize="medium")
    plt.tight_layout()
    plt.show()


plot_ecg(slice(None), f"Full ECG, lead {LEAD}, with important regions")

**Zoomed-in ECG segment.** A short window -- 3.5 to 4 seconds by default -- in finer detail.

In [ ]:
start, end = (int(seconds * SAMPLING_RATE) for seconds in ZOOM_RANGE)
plot_ecg(slice(start, end), f"Zoomed-in ECG, lead {LEAD} ({ZOOM_RANGE[0]:.2f} s - {ZOOM_RANGE[1]:.2f} s)", width=3)

**CXR attribution map.** Pixels above the threshold, marked on the radiograph and coloured
by their importance.

In [ ]:
y_points, x_points = np.where(binary_dilation(cxr_importance >= CXR_THRESHOLD, iterations=1))

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(cxr, cmap="gray")
points = ax.scatter(
    x_points,
    y_points,
    c=cxr_importance[y_points, x_points],
    cmap="summer",
    marker="x",
    s=150,
    vmin=CXR_THRESHOLD,
    vmax=1.0,
    alpha=0.8,
    linewidths=0.7,
)
plt.colorbar(points, ax=ax, fraction=0.04, pad=0.04).set_label(f"Importance ({CXR_THRESHOLD:.2f}-1.00)")
ax.axis("off")
ax.set_title("CXR: important regions", fontsize="x-large")
plt.tight_layout()
plt.show()

## Where next

* The [CardioVAE example](https://github.com/pykale/cardiac/tree/main/examples/cardiovae_multimodal)
  runs this workflow from a configuration file with cross-validation, modality ablation and
  a written report: `python -m examples.cardiovae_multimodal.main --cfg <your config>`.
* The [LS-EMVAE example](https://github.com/pykale/cardiac/tree/main/examples/lsemvae_ecg)
  treats each ECG lead as its own modality.
* The [API overview](https://github.com/pykale/cardiac/blob/main/API_overview.md) lists every
  stage of the KaleCardiac API.